# Introducción a Polars para datos de gran escala

+ Polars es una biblioteca de Python (y también para Rust) ultrarrápida diseñada para manipular y analizar datos estructurados, muy similar a la librería Pandas, pero pensada para datos de gran escala.

+ En la gran mayoría de las tareas de análisis y preparación de datos, Polars es entre 5 y 30 veces más rápido que Pandas, utilizando además una fracción de la memoria RAM.

+ Está diseñado para aprovechar las estructuras de hardware actuales

+ Está escrito/desarrollado en Rust, que es un lenguaje de programación rápido y que usa la memoria de forma más eficiente.

+ Pandas ejecuta la mayoría de sus operaciones en un solo núcleo del procesador.

+ Polars divide el trabajo entre más núcleos disponibles de tu computadora... a.k.a. es multi-threading.

+ Polars tiene una *lazy evaluation* no procesa los datos paso a paso de inmediato si no se le pides. Primero revisa el código, crea un plan de optimización y luego ejecuta todo junto de la forma más eficiente posible.

+ Polars utiliza Apache Arrow en su núcleo, lo que permite un manejo de memoria en columnas superorganizado y sin copias innecesarias de datos.

In [1]:
# Instalación
#pip install polars

In [2]:
import polars as pl
import datetime as dt

In [3]:
df = pl.DataFrame(
    {
        "name": ["Alejandra", "Felipe", "Carlos", "Sofia"],
        "birthdate": [
            dt.date(2001, 1, 10),
            dt.date(1995, 2, 15),
            dt.date(1993, 3, 22),
            dt.date(2001, 4, 30),
        ],
        "weight": [57.9, 72.5, 53.6, 83.1],
        "height": [1.56, 1.77, 1.65, 1.75],
    }
)

print(df)

shape: (4, 4)
┌───────────┬────────────┬────────┬────────┐
│ name      ┆ birthdate  ┆ weight ┆ height │
│ ---       ┆ ---        ┆ ---    ┆ ---    │
│ str       ┆ date       ┆ f64    ┆ f64    │
╞═══════════╪════════════╪════════╪════════╡
│ Alejandra ┆ 2001-01-10 ┆ 57.9   ┆ 1.56   │
│ Felipe    ┆ 1995-02-15 ┆ 72.5   ┆ 1.77   │
│ Carlos    ┆ 1993-03-22 ┆ 53.6   ┆ 1.65   │
│ Sofia     ┆ 2001-04-30 ┆ 83.1   ┆ 1.75   │
└───────────┴────────────┴────────┴────────┘


+ Su uso principal es construir pipelines de datos eficientes para limpiar, transformar y preparar volúmenes de datos de gran escala (desde megabytes hasta cientos de gigabytes)

+ Cargar datos modo eager, i.e. directamente en la memoria: `pl.read_csv()`, `pl.read_parquet()`, `pl.read_ipc()` (formato nativo de Arrow)... aunque estaríamos usando Polars como Pandas y esa no es la idea.

+ La idea es usar el modo lazy de polars, i.e. no cargar el archivo "de golpe".

+ Polars abre una conexión inteligente para procesar el archivo sólo cuando sea estrictamente necesario, permitiendo procesar archivos más grandes que tu memoria RAM.

+ Las correspondientes funciones lazy más importantes son `pl.scan_csv()`, `pl.scan_parquet()`.

+ Y obviamente escribir datos en disco de manera rápida: `df.write_parquet()`, `df.write_csv()`.



+ Recuerden, no hay que pensar en toooooodas las operaciones en DataFrames.

+ Partimos de saber (1) crear nuevas columnas, (2) seleccionar renglones, (3) ordenar con respecto a alguna o algunas variables, hacer agrupaciones (y luego resúmenes de éstas a.k.a agregaciones) y (4) joins.

+ Con estas 4 componentes, el mundo es nuestro!!... Al menos podemos empezar sin mucha fricción.

+ Polars no usa índices (a.k.a. filas numeradas)como Pandas.

+ Para hacer transformaciones de datos, Polars define el concepto de "expression".

+ Las expresions permiten dar flexibilidad y modularidad al proceso de transformación.

In [4]:
pl.col("weight") / (pl.col("height") ** 2)

<Expr ['[(col("weight")) / (col("heigh…'] at 0x7FABAE1E7D10>

In [5]:
otro_df = df.select(
    pl.col("name"), # se selecciona la columna 'name'
    pl.col("birthdate").dt.year().alias("birth_year"), # se selecciona la columna 'birthdate' y se extrae el año
    pl.col("weight"), # se selecciona la columna 'weight'
    pl.col("height"), # se selecciona la columna 'height'
    (pl.col("weight") / (pl.col("height") ** 2)).alias("bmi"), # se crea una nueva columna 'bmi' que es tranformación de otras
)
print(otro_df)

shape: (4, 5)
┌───────────┬────────────┬────────┬────────┬───────────┐
│ name      ┆ birth_year ┆ weight ┆ height ┆ bmi       │
│ ---       ┆ ---        ┆ ---    ┆ ---    ┆ ---       │
│ str       ┆ i32        ┆ f64    ┆ f64    ┆ f64       │
╞═══════════╪════════════╪════════╪════════╪═══════════╡
│ Alejandra ┆ 2001       ┆ 57.9   ┆ 1.56   ┆ 23.791913 │
│ Felipe    ┆ 1995       ┆ 72.5   ┆ 1.77   ┆ 23.141498 │
│ Carlos    ┆ 1993       ┆ 53.6   ┆ 1.65   ┆ 19.687787 │
│ Sofia     ┆ 2001       ┆ 83.1   ┆ 1.75   ┆ 27.134694 │
└───────────┴────────────┴────────┴────────┴───────────┘


In [6]:
df_adicional = df.with_columns(
    birth_year=pl.col("birthdate").dt.year(),
    bmi=pl.col("weight") / (pl.col("height") ** 2),
)
print(df_adicional)

shape: (4, 6)
┌───────────┬────────────┬────────┬────────┬────────────┬───────────┐
│ name      ┆ birthdate  ┆ weight ┆ height ┆ birth_year ┆ bmi       │
│ ---       ┆ ---        ┆ ---    ┆ ---    ┆ ---        ┆ ---       │
│ str       ┆ date       ┆ f64    ┆ f64    ┆ i32        ┆ f64       │
╞═══════════╪════════════╪════════╪════════╪════════════╪═══════════╡
│ Alejandra ┆ 2001-01-10 ┆ 57.9   ┆ 1.56   ┆ 2001       ┆ 23.791913 │
│ Felipe    ┆ 1995-02-15 ┆ 72.5   ┆ 1.77   ┆ 1995       ┆ 23.141498 │
│ Carlos    ┆ 1993-03-22 ┆ 53.6   ┆ 1.65   ┆ 1993       ┆ 19.687787 │
│ Sofia     ┆ 2001-04-30 ┆ 83.1   ┆ 1.75   ┆ 2001       ┆ 27.134694 │
└───────────┴────────────┴────────┴────────┴────────────┴───────────┘


+ `df.with_columns(...)` se usa para crear nuevas columnas o modificar las existentes sin tocar el resto del DataFrame.

+ `df.with_columns(...)` agrega columnas en vez de seleccionarlas. Nótese que tiene las 4 columnas originales de `df` junto con las dos nuevas columnas `birth_year` y `bmi`.

+ `pl.col("nombre").alias("nuevo_nombre")` renombra una columna sobre la marcha mientras hace operaciones.

+ `df.filter(...)` filtra filas basándose en condiciones. Por ejemplo
`df.filter(pl.col("precio") > 100)`.

+ `df.sort(...)` ordena las filas de menor a mayor (o viceversa) según una o varias columnas.

+ `df.drop_nulls()` elimina filas que contenga datos vacíos o faltantes.

+ `df.group_by(...)` agrupa datos según una categoría (por ejemplo, por "país" o por "mes").

+ `df.agg(...)` se usa siempre junto a group_by para calcular métricas como `pl.col("ventas").sum()`, `.mean()`, o `.n_unique()` (contar cuántos elementos distintos hay)

+ `.df.join(...)` fusiona dos tablas distintas basándose en una o varias columnas comúnes (como el ID de un cliente), equivalente al JOIN de SQL.

In [7]:
mi_df = df.filter(pl.col("birthdate").dt.year() < 1998)
print(mi_df)


shape: (2, 4)
┌────────┬────────────┬────────┬────────┐
│ name   ┆ birthdate  ┆ weight ┆ height │
│ ---    ┆ ---        ┆ ---    ┆ ---    │
│ str    ┆ date       ┆ f64    ┆ f64    │
╞════════╪════════════╪════════╪════════╡
│ Felipe ┆ 1995-02-15 ┆ 72.5   ┆ 1.77   │
│ Carlos ┆ 1993-03-22 ┆ 53.6   ┆ 1.65   │
└────────┴────────────┴────────┴────────┘


In [8]:
mi_df = df.filter(
    pl.col("birthdate").is_between(dt.date(1990, 12, 31), dt.date(1997, 1, 1)),
    pl.col("height") > 1.7,
)
print(mi_df)

shape: (1, 4)
┌────────┬────────────┬────────┬────────┐
│ name   ┆ birthdate  ┆ weight ┆ height │
│ ---    ┆ ---        ┆ ---    ┆ ---    │
│ str    ┆ date       ┆ f64    ┆ f64    │
╞════════╪════════════╪════════╪════════╡
│ Felipe ┆ 1995-02-15 ┆ 72.5   ┆ 1.77   │
└────────┴────────────┴────────┴────────┘


In [9]:
mi_df = df.group_by(
    (pl.col("birthdate").dt.year() // 10 * 10).alias("decade"),
    maintain_order=True,
).len()
print(mi_df)

shape: (2, 2)
┌────────┬─────┐
│ decade ┆ len │
│ ---    ┆ --- │
│ i32    ┆ u32 │
╞════════╪═════╡
│ 2000   ┆ 2   │
│ 1990   ┆ 2   │
└────────┴─────┘


In [10]:
mi_df = df.group_by(
    (pl.col("birthdate").dt.year() // 10 * 10).alias("decade"),
    maintain_order=True,
).agg(
    pl.len().alias("sample_size"),
    pl.col("weight").mean().round(2).alias("avg_weight"),
    pl.col("height").max().alias("tallest"),
)
print(mi_df)

shape: (2, 4)
┌────────┬─────────────┬────────────┬─────────┐
│ decade ┆ sample_size ┆ avg_weight ┆ tallest │
│ ---    ┆ ---         ┆ ---        ┆ ---     │
│ i32    ┆ u32         ┆ f64        ┆ f64     │
╞════════╪═════════════╪════════════╪═════════╡
│ 2000   ┆ 2           ┆ 70.5       ┆ 1.75    │
│ 1990   ┆ 2           ┆ 63.05      ┆ 1.77    │
└────────┴─────────────┴────────────┴─────────┘


In [11]:
# Se crea otro DataFrame para hacer un join con el primero
df2 = pl.DataFrame(
    {
        "name": ["Alejandra", "Felipe", "Carlos", "Sofia"],
        "parent": [True, False, False, False],
        "siblings": [1, 2, 3, 4],
    }
)

print(df2)

shape: (4, 3)
┌───────────┬────────┬──────────┐
│ name      ┆ parent ┆ siblings │
│ ---       ┆ ---    ┆ ---      │
│ str       ┆ bool   ┆ i64      │
╞═══════════╪════════╪══════════╡
│ Alejandra ┆ true   ┆ 1        │
│ Felipe    ┆ false  ┆ 2        │
│ Carlos    ┆ false  ┆ 3        │
│ Sofia     ┆ false  ┆ 4        │
└───────────┴────────┴──────────┘


In [12]:
print(df.join(df2, on="name", how="left"))

shape: (4, 6)
┌───────────┬────────────┬────────┬────────┬────────┬──────────┐
│ name      ┆ birthdate  ┆ weight ┆ height ┆ parent ┆ siblings │
│ ---       ┆ ---        ┆ ---    ┆ ---    ┆ ---    ┆ ---      │
│ str       ┆ date       ┆ f64    ┆ f64    ┆ bool   ┆ i64      │
╞═══════════╪════════════╪════════╪════════╪════════╪══════════╡
│ Alejandra ┆ 2001-01-10 ┆ 57.9   ┆ 1.56   ┆ true   ┆ 1        │
│ Felipe    ┆ 1995-02-15 ┆ 72.5   ┆ 1.77   ┆ false  ┆ 2        │
│ Carlos    ┆ 1993-03-22 ┆ 53.6   ┆ 1.65   ┆ false  ┆ 3        │
│ Sofia     ┆ 2001-04-30 ┆ 83.1   ┆ 1.75   ┆ false  ┆ 4        │
└───────────┴────────────┴────────┴────────┴────────┴──────────┘


In [13]:
df3 = pl.DataFrame(
    {
        "name": ["Karen", "Elena", "Gabriel", "Pedro"],
        "birthdate": [
            dt.date(1997, 5, 10),
            dt.date(1985, 6, 23),
            dt.date(2003, 7, 22),
            dt.date(1987, 8, 3),
        ],
        "weight": [67.9, 72.5, 57.6, 93.1],
        "height": [1.76, 1.6, 1.66, 1.8],
    }
)

print(df3)

shape: (4, 4)
┌─────────┬────────────┬────────┬────────┐
│ name    ┆ birthdate  ┆ weight ┆ height │
│ ---     ┆ ---        ┆ ---    ┆ ---    │
│ str     ┆ date       ┆ f64    ┆ f64    │
╞═════════╪════════════╪════════╪════════╡
│ Karen   ┆ 1997-05-10 ┆ 67.9   ┆ 1.76   │
│ Elena   ┆ 1985-06-23 ┆ 72.5   ┆ 1.6    │
│ Gabriel ┆ 2003-07-22 ┆ 57.6   ┆ 1.66   │
│ Pedro   ┆ 1987-08-03 ┆ 93.1   ┆ 1.8    │
└─────────┴────────────┴────────┴────────┘


In [14]:
# Se apilan los DataFrames df y df3
print(pl.concat([df, df3], how="vertical"))

shape: (8, 4)
┌───────────┬────────────┬────────┬────────┐
│ name      ┆ birthdate  ┆ weight ┆ height │
│ ---       ┆ ---        ┆ ---    ┆ ---    │
│ str       ┆ date       ┆ f64    ┆ f64    │
╞═══════════╪════════════╪════════╪════════╡
│ Alejandra ┆ 2001-01-10 ┆ 57.9   ┆ 1.56   │
│ Felipe    ┆ 1995-02-15 ┆ 72.5   ┆ 1.77   │
│ Carlos    ┆ 1993-03-22 ┆ 53.6   ┆ 1.65   │
│ Sofia     ┆ 2001-04-30 ┆ 83.1   ┆ 1.75   │
│ Karen     ┆ 1997-05-10 ┆ 67.9   ┆ 1.76   │
│ Elena     ┆ 1985-06-23 ┆ 72.5   ┆ 1.6    │
│ Gabriel   ┆ 2003-07-22 ┆ 57.6   ┆ 1.66   │
│ Pedro     ┆ 1987-08-03 ┆ 93.1   ┆ 1.8    │
└───────────┴────────────┴────────┴────────┘


## ¿Para qué se usa?

+ ETLs en ingeniería de datos, i.e. extraer datos crudos, transformarlos (limpiar nulos, cambiar formatos) y cargarlos en bases de datos.

+ Procesar datos de gran escala para dejarlos listos antes de entrenar un modelo de ML.

+ Como sustituto de SQL local. Polars es tan rápido que muchas veces es más cómodo y veloz hacer las consultas directamente en Python que montar una base de datos SQL para simplemente analizar "archivos sueltos".

+ El rendimiento no sólo es en términos de velocidad, también es eficiencia.

+ Pandas suele tener el problema de que puede llegar a consumir hasta 3 o 5 veces más RAM que el tamaño real de un archivo.

+ Si se intenta abrir un CSV de 10 GB en una computadora con 16 GB de RAM usando Pandas, lo más probable es que el sistema colapse.

+ Polars soluciona esto al combinar 3 estrategias:

  + El formato Apache Arrow almacena los datos de forma contigua en la memoria. Es muy amigable con el caché del procesador y evita duplicar datos.
  + No hace copias intermedias. En Pandas, cada vez que se encadena una operación (como filtrar y luego renombrar), crea una copia entera de la tabla en la RAM. Polars ejecuta todo en un solo paso optimizado.
  + Si el archivo pesa 50 GB y sólo se tiene 16 GB de RAM, Polars puede procesar el archivo por fragmentos (batches) directo desde el disco. Pandas no puede hacer esto de forma nativa.

+ Ver videito: https://youtu.be/4VS1bPoMnDw?si=UqX4vNYBjvGqjch3